In [11]:
%matplotlib inline

import datacube
import rasterio.features
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datacube.utils.cog import write_cog

import sys
sys.path.insert(1, '../Tools/')
from dea_tools.datahandling import load_ard
from dea_tools.plotting import rgb
from dea_tools.spatial import xr_vectorize


In [12]:
dc = datacube.Datacube(app="Polygonise_landcover_classes")

In [13]:
output_dir = 'vector_classes'
text_size = 35
dpi = 150
lc_product = 'ga_ls_landcover_class_cyear_3'

df = pd.read_csv('input_csv/roi_count_pixels.csv')

# Filter out rows where 'class' is -99
filtered_df = df[df['class'] != -99]

# Create the lists from the filtered DataFrame
lats = filtered_df['centre_y'].tolist()
lons = filtered_df['centre_x'].tolist()
buffers = filtered_df['buffer_size_wgs84'].tolist()
times = filtered_df.apply(lambda row: (str(row['start_year']), str(row['end_year'])), axis=1).tolist()
intervals = filtered_df['interval'].tolist()
class_codes = filtered_df['class'].tolist()

roi_names = filtered_df['name'].tolist()

In [15]:
for lat, lon, buffer, time, interval, roi_name, class_code in zip(lats, lons, buffers, times, intervals, roi_names, class_codes):
    gdfs = []
    
    lat_range = (lat - buffer, lat + buffer)
    lon_range = (lon - buffer, lon + buffer)
    
    query ={
        'x': lon_range,
        'y': lat_range,
        'time': time
    }
    ds_lc = dc.load(product=lc_product,
                measurements=['level3'],
                **query)

    ds_lc_lvl3 = ds_lc.level3 

    for time in ds_lc_lvl3.time:
        start_year = str(time.data)[:4]
        end_year = str(int(start_year)+1)
        ds = ds_lc_lvl3.sel(time=time)
        
        gdf = xr_vectorize(da = ds, 
                           mask = ds.values==class_code,
                           attribute_col='landcover_class_code')
       
        # take individual gdf's and combine them, including the year as an attribute. Needed for xr_animation
        gdf['start_time']=start_year
        gdf['end_time']=end_year
        gdfs.append(gdf)

    combined_gdfs = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
    combined_gdfs.crs = gdfs[0].crs
    combined_gdfs.to_file(f'{output_dir}/{roi_name}_lc_lvl3_combined_years.geojson', driver='GeoJSON')